In [ ]:
!pip install -q dbt-duckdb
import duckdb
duckdb.sql("""
           force install avro from core_nightly;
           force install azure from core_nightly;
           force install iceberg from core_nightly;
           """)

In [ ]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    lakehouse_name = vl.lakehouse_name
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
    token          = notebookutils.credentials.getToken('storage')
    dbt_target     = 'dev'
    notebookutils.fs.cp(
        f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/dbt',
        '/tmp',
        True,
    )
    dbt_path = '/tmp/dbt'
except ModuleNotFoundError:
    from azure.identity import AzureCliCredential
    import yaml
    from pathlib import Path
    _root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "deploy_config.yml").exists()), None)
    if _root is None:
        raise FileNotFoundError("deploy_config.yml not found in cwd or any parent — run from inside the cloned repo")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws"]
    lakehouse_id   = _cfg["lakehouse"]
    lakehouse_name = _cfg["lakehouse_name"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    token          = AzureCliCredential().get_token("https://storage.azure.com/.default").token
    dbt_target     = 'dev'
os.environ['download_limit']   = download_limit
os.environ['process_limit']    = process_limit

In [ ]:
os.environ['FILES_PATH']       = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files'
os.environ['WAREHOUSE_PATH']   = f'{workspace_id}/{lakehouse_id}'
os.environ['ONELAKE_ENDPOINT'] = 'https://onelake.table.fabric.microsoft.com/iceberg'
os.environ['ONELAKE_TOKEN']    = token

In [ ]:
from dbt.cli.main import dbtRunner
os.chdir(dbt_path)
dbt = dbtRunner()
result = dbt.invoke(["run", "--target", dbt_target, "--profiles-dir", "."])
if not result.success:
    print("dbt run had failures — retrying failed models once...")
    _ = dbt.invoke(["retry", "--target", dbt_target, "--profiles-dir", "."])
_ = dbt.invoke(["test", "--target", dbt_target, "--profiles-dir", "."])